# Jury Rank Similarity — ESC2024 Grand Final

For each pair of national juries, compute how similar their top-N rankings are using the Jaccard index. Plot results as heatmaps for thresholds 2 through 10, then a high-resolution one for top-10. Finally, list which countries have a high overlap with selected base countries (e.g. which juries' top-N share at least N matches with Spain's top 3).

## Section 1 — Setup

In [ ]:
# Widen Jupyter output for the heatmap and tables.
from IPython.display import display, HTML
display(HTML("<style>.container { width:95% !important; }</style>"))


In [ ]:
import pandas as pd

# Pandas 2.x compatibility shim for legacy DataFrame.append usage
if not hasattr(pd.DataFrame, "append"):
    def _df_append(self, other, ignore_index=False, verify_integrity=False, sort=False):
        if isinstance(other, pd.DataFrame):
            to_concat = [self, other]
        else:
            to_concat = [self, pd.DataFrame([other])]
        return pd.concat(to_concat, ignore_index=ignore_index, verify_integrity=verify_integrity, sort=sort)
    pd.DataFrame.append = _df_append
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import os
from scipy.cluster.hierarchy import dendrogram, linkage, leaves_list

### File paths

In [ ]:
jury_voting_csv = "ESC2024-GF_Jury_Ranking_NB.csv"
intput_folder = "./Input"
output_folder = "./Output"

## Section 2 — Load and prepare the rank matrix

Load the GF jurors CSV (rows = countries, columns = the rank each jury gave each country). Replace `0` (no rank assigned) with `26` so it sorts to the bottom.

In [ ]:
# Load dataset with index  = Country
gf_jurors_df = pd.read_csv(os.path.join(intput_folder, jury_voting_csv), sep=";", index_col='Country')
gf_jurors_df = gf_jurors_df.drop(columns=["CountryId"])
gf_jurors_df = gf_jurors_df.replace(0, 26)
display(gf_jurors_df)

### Transpose: rows = positions, columns = juries

For each voting jury, list the countries in rank order (1st through 26th).

In [ ]:
# Create a new DataFrame for rearranged results
gf_jurors_df_T = gf_jurors_df.T
result_df = pd.DataFrame(index=['1st Place', '2nd Place', '3rd Place', "4th Place", "5th place", "6th Place", "7th Place"
                                , "8th Place", "9th Place","10th Place", "11th Place", "12th Place", "13th Place", "14th Place", "15th Place"
                                ,'16th  Place', '17th  Place', '18th  Place', "19th Place", "20th place", "21st Place", "22nd Place"
                                , "23rd Place", "24th Place","25th Place","26th Place"])

# Process each column to transform rankings to country names
for voter in gf_jurors_df_T.columns:
    # Sort the column to get ranks; lower scores are better, so we sort ascending
    sorted_countries = gf_jurors_df_T[voter].sort_values().index
    result_df[voter] = sorted_countries[0:]
    
display(result_df)

## Section 3 — Jaccard similarity functions and matrix

Three helpers:
- `jaccard_index(set1, set2)` — standard Jaccard.
- `jaccard_index_own(set1, set2)` — variant divided by `len(set1)` only (asymmetric).
- `calculate_jaccard(df_input, option_own)` — pairwise matrix across columns of a binary dataframe.

Plus `get_matching_countries(...)` further down for the per-country lookup at the bottom.

In [ ]:
def jaccard_index(set1, set2):
    '''
    Calculate the jaccard index between 2 arrays.
    '''
    intersection = len(set1.intersection(set2))
    union = len(set1.union(set2))
    if union == 0:  # prevent division by zero
        return 0
    return intersection / union

def jaccard_index_own(set1, set2):
    '''
    Calculate the jaccard index between 2 arrays.
    '''
    intersection = len(set1.intersection(set2))
    #union = len(set1.union(set2))
    if len(set1) == 0:  # prevent division by zero
        return 0

    return intersection / len(set1)


def calculate_jaccard(df_input, option_own):
    '''
    Calculate all the jaccard indexes in a matrix.
    '''
    n = len(df_input)
    jaccard_df = pd.DataFrame(0, index=df_input.index, columns=df_input.index, dtype=float)
    
    voted_countries = df_input.columns.values

    # Calculate Jaccard Index for each pair of voters
    for i in range(n):
        for j in range(i+1, n):
            # Get countries voted by each voter
            votes1 = set(df_input.columns[df_input.iloc[i] == 1])
            votes2 = set(df_input.columns[df_input.iloc[j] == 1])
            

            
            if df_input.index[i] in voted_countries:
                votes1.add(df_input.index[i])
            if df_input.index[i] in voted_countries:
                votes2.add(df_input.index[j])
            '''
            if i ==1:
                print(df_input.index[i],df_input.index[j],votes1, votes2 )
            '''
            


            # Calculate Jaccard Index
            if option_own == "True":
                jaccard = jaccard_index_own(votes1, votes2)
            else :
                jaccard = jaccard_index(votes1, votes2)

            # Store the result in the DataFrame symmetrically
            jaccard_df.iloc[i, j] = jaccard
            jaccard_df.iloc[j, i] = jaccard

    # Fill diagonal with 1s since a set is always perfectly similar to itself
    np.fill_diagonal(jaccard_df.values, 1)

    return jaccard_df


def replace_unallowed_strings(df, allowed_strings):
    '''
    Replace individual cell values.
    '''
    def replace(value):
        return value if value in allowed_strings else ""
    
    return df.applymap(replace)

def get_matching_countries(df_all, base_country, nb_place_looked, match_threshold, fill_option, base_country_opt):
    '''
    Extract the top nb_place_looked choices for the base country & Prepare a dictionary to eventually create the DataFrame.
    '''
    base_top_votes = df_all[base_country].head(nb_place_looked).tolist()
    if base_country_opt == 'True':
        base_top_votes.append(base_country)
    result_data = {}
    
    # Iterate over all countries in the DataFrame columns
    for country in df_all.columns:
        if country == base_country:
            continue  # Skip the base country
        
        # Get top votes for the current country & Find common elements
        country_top_votes = df_all[country].head(nb_place_looked).tolist()
        common_votes = set(base_top_votes).intersection(set(country_top_votes))

        if len(common_votes) >= match_threshold:      
            result_data[country] = df_all[country].head(nb_place_looked).tolist()
    
    # Creating the DataFrame from the dictionary & Insert the base_country column as the first column
    result_df = pd.DataFrame.from_dict(result_data, orient='index', columns=[f'{i+1}st Place' for i in range(nb_place_looked)]).transpose()
    result_df.insert(0,base_country, df_all[base_country].head(nb_place_looked).tolist())
    result_df = replace_unallowed_strings(result_df, base_top_votes)
    
    return result_df

### Compute one Jaccard matrix at threshold 10 (initial check)

In [ ]:
df_binary = gf_jurors_df.applymap(lambda x: 0 if (x > threshold) & (x !=0) else 1)
correlation_matrix = calculate_jaccard(df_binary, option_own = "False")

## Section 4 — Heatmaps

### 4.1 Per-threshold grid (top 2 through top 10)

In [ ]:
# Define the range of thresholds to consider, excluding threshold = 1
thresholds = range(2, 11)

# Set up the matplotlib figure and define dimensions for a 3x3 grid
fig, axes = plt.subplots(nrows=3, ncols=3, figsize=(80, 70))  # Adjusted figure size
fig.subplots_adjust(hspace=0.25, wspace=0.1)  # Adjusted spacing
axes = axes.flatten()

# Loop through threshold values from 2 to 10
for i, threshold in enumerate(thresholds):
    # Replace values above the threshold with 0
    #rank_table_0 = rank_table.where(rank_table <= threshold, 0)
    df_binary = gf_jurors_df.applymap(lambda x: 0 if (x > threshold) & (x !=0) else 1)
    correlation_matrix = calculate_jaccard(df_binary, option_own = "False")
    linked = linkage(correlation_matrix, 'single')
    order = leaves_list(linked)
    ordered_corr_matrix = correlation_matrix.iloc[order, order]

    # Plot the heatmap in each subplot
    ax = axes[i]
    sns.set(font_scale=2)
    sns.heatmap(ordered_corr_matrix, annot=False, cmap='coolwarm', fmt=".1f", 
                linewidths=.2, linecolor='black', vmin=0, vmax=1, cbar_kws={"shrink": .6}, ax=ax)
    sns.set(font_scale=2.5)
    ax.set_title(f'Top {threshold} countries', fontsize=50)
    ax.set_xlabel('National Juries', fontsize=2)
    ax.set_ylabel('National Juries', fontsize=2)

    
plt.gcf().set_facecolor('white')  # for the figure
plt.gca().set_facecolor('whitesmoke')  # for the axes
# Adjust the overall figure settings
plt.suptitle('Similarity between National Juries ranks - ESC2024_GF', fontsize=80)
#plt.show()
plt.savefig(os.path.join(output_folder,"ESC2024_GF_Juries_Similarity.jpg"), dpi=450, bbox_inches='tight')

### 4.2 Standalone top-10 heatmap (higher resolution)

In [ ]:
threshold = 10
df_binary = gf_jurors_df.applymap(lambda x: 0 if (x > threshold) & (x !=0) else 1)
correlation_matrix = calculate_jaccard(df_binary, option_own = "False")
linked = linkage(correlation_matrix, 'single')
order = leaves_list(linked)
ordered_corr_matrix = correlation_matrix.iloc[order, order]
# Customize the figure size
plt.figure(figsize=(20, 16))
#mask = np.triu(np.ones_like(correlation_matrix, dtype=bool))
mask = np.eye(len(correlation_matrix), dtype=bool)
# Plot the heatmap
sns.set(font_scale=1.5)
sns.heatmap(ordered_corr_matrix, annot=False, cmap='coolwarm', fmt=".2f", 
            linewidths=.05, linecolor='black', vmin=0, vmax=1,cbar_kws={"shrink": .8})

# vmin=0, vmax=1,
# Rotate labels if necessary
plt.xticks(rotation=90)
plt.yticks(rotation=0)

# Set labels and title with custom font size
plt.xlabel('National juries', fontsize=20)
plt.ylabel('National juries', fontsize=20)
plt.title('Similarity between National Juries ranks for top 10 ranks - ESC2024_GF', fontsize=26)

# Set background color
plt.gcf().set_facecolor('white')  # for the figure
plt.gca().set_facecolor('whitesmoke')  # for the axes

#plt.show()
plt.savefig(os.path.join(output_folder, "ESC2024_GF_Juries_Similarity_Top10.jpg"), dpi=300, bbox_inches='tight')
#plt.savefig('heatmap_voting_analysis_top10.png', dpi=300, bbox_inches='tight')

## Section 5 — Per-country matching analysis

For selected base countries, list which other juries had at least `match_threshold` countries in common with the base country's top `nb_place_looked` choices. Each cell below targets a specific country and is meant to be edited or commented out individually.

In [ ]:
df_Belgium_matching_top2= get_matching_countries(df_all = result_df, base_country = "Spain", nb_place_looked = 3, match_threshold = 2
                                            , fill_option = 'None',base_country_opt= "True")
display(df_Belgium_matching_top2)


In [ ]:
df_Czechia_matching_top10 = get_matching_countries(df_all = result_df, base_country = "Czechia", nb_place_looked = 11, match_threshold = 7
                                            , fill_option = 'None',base_country_opt= "True")
display(df_Czechia_matching_top10)


In [ ]:
df_malta_matching_top4 = get_matching_countries(df_all = result_df, base_country = "Malta", nb_place_looked = 5, match_threshold = 4
                                            , fill_option = 'None',base_country_opt= "True")
display(df_malta_matching_top4)
#df_malta_matching_top9.to_excel(os.path.join(output_folder, "malta_matching_top9.xlsx"))


In [ ]:
df_Australia_matching_top4 = get_matching_countries(df_all = result_df, base_country = "Australia", nb_place_looked = 5, match_threshold = 3 
                                            , fill_option = 'None',base_country_opt= "True")
display(df_Australia_matching_top4)
#df_armenia_matching_top10.to_excel(os.path.join(output_folder, "armenia_matching_top10.xlsx"))


In [ ]:
df_Ukraine_matching_top8 = get_matching_countries(df_all = result_df, base_country = "Ukraine", nb_place_looked = 8, match_threshold = 7
                                            , fill_option = 'None',base_country_opt= "True")
display(df_Ukraine_matching_top8)
#df_Ukraine_matching_top8.to_excel(os.path.join(output_folder, "Ukraine_matching_top8.xlsx"))


In [ ]:
df_Germany_matching_top7 = get_matching_countries(df_all = result_df, base_country = "Germany", nb_place_looked = 6, match_threshold = 5
                                            , fill_option = 'None',base_country_opt= "True")
display(df_Germany_matching_top7)
#df_georgia_matching_top6.to_excel(os.path.join(output_folder, "georgia_matching_top6.xlsx"))


In [ ]:
df_Iceland_matching_top9 = get_matching_countries(df_all = result_df, base_country = "Iceland", nb_place_looked = 10, match_threshold = 8
                                            , fill_option = 'None',base_country_opt= "True")
display(df_Iceland_matching_top9)
#df_georgia_matching_top6.to_excel(os.path.join(output_folder, "georgia_matching_top6.xlsx"))


In [ ]:
df_UK_matching_top7 = get_matching_countries(df_all = result_df, base_country = "United Kingdom", nb_place_looked = 7, match_threshold = 5
                                            , fill_option = 'None',base_country_opt= "True")
display(df_UK_matching_top7)
#df_armenia_matching_top10.to_excel(os.path.join(output_folder, "armenia_matching_top10.xlsx"))


In [ ]:
df_Poland_matching_top4 = get_matching_countries(df_all = result_df, base_country = "Poland", nb_place_looked = 5, match_threshold = 3
                                            , fill_option = 'None',base_country_opt= "True")
display(df_Poland_matching_top4)


In [ ]:
df_Latvia_matching_top6 = get_matching_countries(df_all = result_df, base_country = "Latvia", nb_place_looked = 6, match_threshold = 5
                                            , fill_option = 'None',base_country_opt= "True")
display(df_Latvia_matching_top6)


In [ ]:
df_Georgia_matching_top10 = get_matching_countries(df_all = result_df, base_country = "Georgia", nb_place_looked = 10, match_threshold = 8
                                            , fill_option = 'None',base_country_opt= "True")
display(df_Georgia_matching_top10)


In [ ]:
df_Armenia_matching_top5 = get_matching_countries(df_all = result_df, base_country = "Armenia", nb_place_looked = 5, match_threshold = 5
                                            , fill_option = 'None',base_country_opt= "True")
display(df_Armenia_matching_top5)


In [ ]:
df_Moldova_matching_top5 = get_matching_countries(df_all = result_df, base_country = "Moldova", nb_place_looked = 5, match_threshold = 4
                                            , fill_option = 'None',base_country_opt= "True")
display(df_Moldova_matching_top5)


## Run summary

In [ ]:
print("=" * 60)
print(f"Heatmap analysis complete — input: {jury_voting_csv}")
print(f"Output saved to: {output_folder}")
print(f"  ESC2024_GF_Juries_Similarity.jpg          (top 2-10 grid)")
print(f"  ESC2024_GF_Juries_Similarity_Top10.jpg    (top 10 standalone)")
print("=" * 60)
